# Finetuning Nemotron 3 Nano in NeMo Automodel
This tutorial will detail instructions for finetuning Nemotron 3 Nano with SFT and PEFT using Automodel on a single H100 GPU. For this example, we will be finetuning on the SQuAD (Stanford Question Answering Dataset) dataset with **Parameter-Efficient Fine-Tuning (PEFT)**. **This example shows PEFT but the commands are similar and included for SFT**.

This notebook is intended to show the steps to finetune and also explain the key concepts of the NeMo Automodel framework, including recipes, YAML configuration files, and how to use the finetuned models after.

## What is Automodel

Automodel is part of the NeMo framework and is a PyTorch based library for pretraining and finetuning. **It is Hugging Face compatible so there is never a need to do any checkpoint conversions.** The result of using Automodel will be a fully interoperable checkpoint with the rest of the HF ecosystem you are able to drop the finetuned model into another tool as expect it to work as is. 

As a rule of thumb, Automodel is best used when a model is available on Hugging Face and the scale is less than 1000 GPUs. Otherwise, it is recommended to use Megatron-bridge.

## Requirements

- Access to the latest NeMo Automodel NGC container
- Access to A100 80GB GPUs
- A valid Hugging Face API token

## Step 0. Launch the NeMo Framework container

Run the following command to launch the NeMo Automodel container. Ensure to populate the `HF_TOKEN` variable with a valid API key:

```python
docker run -it \
    -p 8080:8080 -p 8088:8088 \
    --rm --gpus all --shm-size=8g \
    --ipc=host -v $(pwd):/workspace \
    -e HF_TOKEN=<YOUR_HF_TOKEN> \
    nvcr.io/nvidian/nemo-automodel:26.02.rc0 /bin/bash

#### Launch Jupyter Notebook as follows:
```python
jupyter notebook --allow-root --ip 0.0.0.0 --port 8088 --no-browser --NotebookApp.token=''

## Datasets

### Understanding the Data Format

All HF tranformer models expect data in the format:
```
{
    'input_ids' : [1, 234, 456, 789, ...], # Tokenized text
    'labels': [-100, -100, -100, 789, ...], # Training target
    'attention_mask': [1, 1, 1, 1, ...] # Real tokens (1) vs padding (0)
}
```

```input_ids```: The tokenized version of your text (prompt + answer)
- Example: "What is AI? Answer: Artificial Intelligence"
- Becomes: `[1, 1867, 318, 9552, 30, 23998, 25, 3433, 12345]`

```labels```: Same as input_ids but with prompt tokens masked as -100
- Only the answer tokens contribute to the loss
- Prompt tokens are set to -100 (ignored by the loss function)
- Without masking, the model learns to predict everything (prompt + answer) which is not ideal for instruction following + wastest training on repeating the question

```attention_mask```: Indicates which tokens are real vs padding
- `1`: real token (attend to it)
- `0`: padding token (ignore it)

### Types of Datasets
#### Completion Datasets
Completion datasets are single text sequences designed for language modeling where the model learns to predict the next token given a context. These datasets typically contain a context (prompt) and a target (completion). 

Example:
**HellaSwag**:
- Context (ctx): A situation or scenario description
- Endings: Multiple possible completions (4 options)
- Label: Index of the correct ending

```
Context: "A man is sitting at a piano in a large room."
Endings: [
  "He starts playing a beautiful melody.",
  "He eats a sandwich while sitting there.",
  "He suddenly becomes invisible.",
  "He transforms into a robot."
]
Label: 0  # First ending is correct
```

NeMo Automodel provides the SFTSingleTurnPreprocessor class to handle completion datasets. This processor:
1. Extracts context and target using get_context() and get_target().
2. Tokenizes and cleans context and target separately.
3. Concatenates them into one sequence.
4. Creates loss mask: -100 for context, target IDs for target.
5. Pads sequences to equal length.

To create your own completion dataset, define a class like
```python
from datasets import load_dataset
from nemo_automodel.components.datasets.utils import SFTSingleTurnPreprocessor

class MyCompletionDataset:
    def __init__(self, path_or_dataset, tokenizer, split="train"):
        raw_datasets = load_dataset(path_or_dataset, split=split)
        processor = SFTSingleTurnPreprocessor(tokenizer)
        self.dataset = processor.process(raw_datasets, self)

    def get_context(self, examples):
        """Extract context/prompt from your dataset"""
        return examples["context_field"]  # Replace with your context field

    def get_target(self, examples):
        """Extract target/completion from your dataset"""
        return examples["target_field"]   # Replace with your target field

    def __getitem__(self, index):
        return self.dataset[index]

    def __len__(self):
        return len(self.dataset)
```

#### Instruction Datasets
Instruction datasets are question-answer pairs where the model learns to respond to specific instructions or questions. These datasets are structured as context-question pairs with corresponding answers, making them ideal for teaching models to follow instructions and provide accurate responses.

Example:
**SQuAD**

To create you own instruction dataset, follow the make_squad_dataset pattern:
```python
from datasets import load_dataset

def make_my_instruction_dataset(
    tokenizer,
    seq_length=None,
    limit_dataset_samples=None,
    split="train",
    dataset_name="your-dataset-name",
):
    if limit_dataset_samples:
        split = f"{split}[:{limit_dataset_samples}]"

    dataset = load_dataset(dataset_name, split=split)

    return dataset.map(
        your_own_fmt_fn,  # Your formatting function
        batched=False,
        remove_columns=dataset.column_names,
    )
```


### SQuAD

In this tutorial, we fine-tune on the [SQuAD](https://rajpurkar.github.io/SQuAD-explorer/) (Stanford Question Answer Dataset). SQuAD is a **reading comprehension dataset** that is composed of questions on a set of Wikipedia articles where the answer to every question is a segment of text, or span, from the corresponding reading passage, or the question might be unanswerable. 

In this tutorial, we will be using SQuAD v1.1 where all answers are guaranteed to be present in the context. 

**Original Data**:

```json
{

    "id": "5733be284776f41900661182",
    "title": "University_of_Notre_Dame",
    "context": "Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend Venite Ad Me Omnes. Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.",
    "question": "To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?",
    "answers": {
        "text": [
            "Saint Bernadette Soubirous"
        ],
        "answer_start": [
            515
        ]
    }
}
```

For SQuAD, Automodel provides the make_squad_dataset function which formats and prepares the dataset.

**After processing with the nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 tokenizer's chat template, the decoded text is:**

**input_id**

```python
<|im_start|>system
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.<|im_end|>
<|im_start|>user
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|im_end|>
<|im_start|>assistant
<think></think>Saint Bernadette Soubirous<|im_end|>
```

**labels (non-masked tokens only)**
```python
<|im_start|>assistant
<think></think>Saint Bernadette Soubirous<|im_end|>
<|im_end|>
```


#### nemo_automodel/components/datasets/llm/squad.py

```python
def make_squad_dataset(
    tokenizer, # HF tokenizer with attributes eos_token_id, optional: bos_id, eos_id, chat_template, apply_chat_template
    seq_length=None, # if set, pad/truncate each example to this length
    limit_dataset_samples=None, # if set, limit the number of examples loaded from the split
    fp8=False, # flag for future use, currently unused
    split="train", # which split of the dataset to use (train or validation)
    dataset_name="squad", # identifier for the HF dataset (ex: rejpurkar/squad)
    padding=False, # optional padding strategy
    truncation=False, # optional truncation strategy
):

# uses HF slicing syntax to limit dataset size
if limit_dataset_samples is not None: 
        assert isinstance(limit_dataset_samples, int), "Expected limit_dataset_samples to be an int"
        if not "[" in split:
            split = f"{split}[:{limit_dataset_samples}]"
        else:
            logging.warning(f"Dataset split {split} already has a slice, skipping limit_dataset_samples")
    dataset = load_dataset(dataset_name, split=split) # load raw dataset from HF

    # format the dataset
    chat_template = getattr(tokenizer, "chat_template", None) # checks if tokenizer has chat template
    eos_token_id = getattr(tokenizer, "eos_token_id", 0) # gets the EOS token ID

    # if pad_token_id is not set, use eos_token_id
    # therefore, pad_token can either [PAD] or [EOS]
    pad_token_id = _add_pad_token(tokenizer) or eos_token_id # adding pad token if not present

    if chat_template is None: # does this model know how to handle conversations?

        # chat template is special formatting string that instruction tuned models use to structure conversations
        fmt_fn = lambda x: _formatting_prompts_func( # GPT-2, Original LLama, Bert, most base models
            x, tokenizer, eos_token_id, pad_token_id, seq_length, padding, truncation
        ) # this is doing simple string concatenation 
    else:

        # uses model-specific special tokens to structure the conversation
        # format it into a chat template which is in the tokenizer
        fmt_fn = lambda x: _formatting_prompts_func_with_chat_template( # Llama-3-Instuct, Mistral-instruct, Gemma IT
            x, tokenizer, eos_token_id, pad_token_id, seq_length, padding, truncation
        )  # noqa: E731

    # map the dataset
    # this is converting raw data into training ready format
    return dataset.map( 
        fmt_fn,
        batched=False,
        remove_columns=["id", "title", "context", "question", "answers"],
    )
```

**_formatting_prompts_func**
Wil concatenate context, question, answer into a single string

**_formatting_prompts_func_with_chat_template**
Will format the context, question, answer into a chat template

### Other Data Formatting Functions

Other functions provided by Automodel can be found in **nemo_automodel/components/datasets/llm**. Two that might be more useful are:

1. **ColumnMappedTextInstructionDataset** (nemo_automodel/components/datasets/llm/column_mapped_text_instruction_dataset.py)
- This is the most felxible option and works with any dataset that has 2-3 text columns. 
- The types of column combinations include:
    - Question + Answer
    - Context + Answer
    - Context + Questions + Answer
- Can work with both HF datasets and local JSON files
- Also requires a column mapping in the config

Ex:
```yaml
dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: "/tmp/simple_qa.jsonl"
  column_mapping:
    question: query        # Maps 'query' → 'question'
    answer: response       # Maps 'response' → 'answer'
  answer_only_loss_mask: true
```

##### Valid Column Combinations

| Fields | Mapping | Use Case |
|--------|---------|----------|
| **2** | `question` + `answer` | Q&A, instruction following |
| **2** | `context` + `answer` | Summarization, rewriting |
| **3** | `context` + `question` + `answer` | Reading comprehension |


2. **ChatDataset** (nemo_automodel/components/datasets/llm/chat_dataset.py)
- If the data is in OpenAI-format tool-calling chat transcripts. The class expects each row to contain a 'messages' list in OpenAI chat format. This can potentially include tool calls and tool responses. 




If your dataset has more than 3 columns, you will need to preprocess it first. For example, you can combine multiple fields into a structured prompt. After preprocessing, you can use ColumnMappedTextInstructionDataset. If your data has complex formatting needs, you can write a custom dataset function. 

## Use a Recipe to Fine-Tune the Model

### What is a **recipe**?
A recipe is a self-contained module that defines how the training actually happens. They contain the training logic and orchestration. The recipe class reads the YAML configuration and instantiates all the components. It then implements the training loop, handled checkpointing, manages distributed training, logs metrics, and runs validation.

**Example - nemo_automodel/recipes/llm/train_ft.py**
```python
class TrainFinetuneRecipeForNextTokenPrediction(BaseRecipe):
    """Recipe for fine-tuning a model for next-token prediction.

    This class orchestrates training, from setup to main training loop.
    """

    def __init__(self, cfg):
        """Initialize the recipe with configuration.

        Args:
            cfg: Configuration dictionary/object for training.
        """
        self.cfg = cfg

### What is the **YAML file**?
The yaml file is the configuration file that specified what you want to train. They contain all the hyperparameters and settings for a training run. The yaml file declares things like which model to use, learning rate, batch size, optimizer settings, PEFT configuration, dataset settings, distributed training settings, etc.. 

#### Understanding the YAML file 
The `_target_` pattern tells the framework which Python class or function to instantiate. When the recipe calls .instantiate() on a config section, it will import the class/function specified in `_target_` and parse the other keys as arguments to that class/function.

**Example**

```python
optimizer:
  _target_: torch.optim.Adam
  betas: [0.9, 0.999]
  eps: 1e-8
  lr: 1.0e-5
  weight_decay: 0
```

This becomes equivalent to 
```python
optimizer = torch.optim.Adam(params=model_params, betas=[0.9,0.999], eps=1e-8, lr=1.0e-5, weight_decay=0)

If the recipe calls .instantiate(), then `_target_` is needed on that config section. For example, in train_ft.py, 
```python
self.peft_config = self.cfg.peft.instantiate() # <- needs _target_ in YAML

self.step_scheduler = StepScheduler(**cfg.step_scheduler.to_dict()) # <- no _target_ needed
``` 


#### **YAML file deepdive** -- examples/llm_finetune/nemotron/nemotron_nano_v3_squad.yaml 
```python

# This controls the training loop -- how many steps to train, when to save checkpoints, when to validate. It also manages the gradient accumulation. Default settings (ex: num_epochs=10, global_batch_size=32, etc.) can be found in the recipe file (train_ft.py).
step_scheduler:
  global_batch_size: 16 # total samples across all GPUs per step
  local_batch_size: 1 # samples per GPU per forward pass
  ckpt_every_steps: 1000 # Save checkpoint every N steps
  val_every_steps: 1000  # will run validation every x number of gradient steps
  max_steps: 100 # stop training after this many steps

# Configured the low-level communication layer for multi-GPU training. Not needed for single GPU training.
dist_env:
  backend: nccl # NVIDIA collective communications library
  timeout_minutes: 1 # how long to wait for GPU communication before considering it a failure. In minutes.

# Random Number Generator for reproducibility and checkpointing
rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true # Each GPU rank gets a different but deterministic seed

# Specified which pretrained model to load and fine-tune. This should be the name of any Hugging Face model.
model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16

# Enables torch.compile configuration for faster training by compiling the model into optimized kernels.
compile:
  enabled: false
  mode: "default"  # Options: "default", "reduce-overhead", "max-autotune"
  fullgraph: false
  dynamic: true  # Set to false for better performance with fixed shapes
  backend: null  # Use default backend (inductor)

# Configures multi-GPU training strategy. FSDP shards the model across GPUs, TP splits individual layers, and CP splits long sequences.
distributed:
  _target_: nemo_automodel.components.distributed.fsdp2.FSDP2Manager
  dp_size: none
  dp_replicate_size: 1 # dp_shard_size = dp_size / dp_replicate_size and dp_shard_size < dp_size. For DDP usecase, use DDPManager
  tp_size: 1 # ARTI: 8 --> see if SFT works on workstation
  cp_size: 1
  sequence_parallel: false
  defer_fsdp_grad_sync: false

# Defines the loss function that will be minimized during training.
loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

# Specified which dataset to use for training and where to load it from. For the SQuAD dataset, NeMo Automodel provides the make_squad_dataset function which formats the prepares the dataset (e.g., formatting).
dataset:
  _target_: nemo_automodel.components.datasets.llm.squad.make_squad_dataset
  dataset_name: rajpurkar/squad
  split: train

# Packs multiple short training examples into one long sequence to maxmimize GPU utilization. 0 = disabled.
packed_sequence:
  packed_sequence_size: 0

# Controls how data is batched and fed to the model. Handles shuffling, parallel loading, and data preprocessing.
dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  shuffle: True

# Specifies evaluation data.
validation_dataset:
  _target_: nemo_automodel.components.datasets.llm.squad.make_squad_dataset
  dataset_name: rajpurkar/squad
  split: validation
  limit_dataset_samples: 64

# Same as training dataloader but for the validation data.
validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater

# Defines the optimization algorithm.
optimizer:
  _target_: torch.optim.Adam
  betas: [0.9, 0.999]
  eps: 1e-8
  lr: 1.0e-5
  weight_decay: 0

# This adjusts the learning rate during training. 
lr_scheduler:
  lr_decay_style: cosine
  min_lr: 1.0e-6

# wandb:
#   project: <your_wandb_project>
#   entity: <your_wandb_entity>
#   name: <your_wandb_exp_name>
#   save_dir: <your_wandb_save_dir> 

**For PEFT**
```yaml
peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig # Using the LoRA implementation from NeMo Automodel
  match_all_linear: True # applies LoRA adapters to all linear/matrix-multiplication layers. If False, provide target_modules like []"q_proj", "v_proj"]
  dim: 8 # sets the rank of the low-rank decomp
  alpha: 32 # scaling factor 
  use_triton: True # uses Triton-optimized GPU kernels for LoRA operations 
```

### Under the hood

**examples/llm_finetune/finetune.py**

This is a entry point script that will load the YAML config file, instantiate the recipe, and run the training by calling setup() and run_train_validation_loop().

```python
def main(default_config_path="examples/llm_finetune/llama3_2/llama3_2_1b_hellaswag.yaml"):
    """Main entry point for the fine-tuning recipe.

    Loads the configuration, sets up the recipe, and initiates the training loop.
    """
    # Step 1: Reads the YAML config file, merges command-line arguments with YAML config, returns a 
    # ConfigNode object that contains all the training settings
    cfg = parse_args_and_load_config(default_config_path) 

    # Step 2: Instantiating the recipe class and passes the configuration to the recipe. At this point, 
    # nothing is built yet. This is just a container with the config.
    recipe = TrainFinetuneRecipeForNextTokenPrediction(cfg)

    # Step 3: The recipe builds all training components (distributed environment, loads the pretrained model, 
    # adds the LoRA adapter, creates the optimizer, creates training data pipeline, sets up logging, etc.)
    recipe.setup()

    # Step 4: Runs the training loop until completion
    recipe.run_train_validation_loop()
```

## Run the PEFT recipe

The model checkpoint will be saved under the checkpoints/ directory. For PEFT, it will have the following contents:

```
checkpoints/epoch_0_step_99/
├── config.yaml
├── step_scheduler.pt
├── losses.json
├── dataloader
│   ├── dataloader_dp_rank_0.pt
├── model
│   ├── configuration_nemotron_h.py
│   ├── tokenizer.json
│   ├── tokenizer_config.json
│   ├── automodel_peft_config.json
│   ├── adapter_model.safetensors
│   ├── chat_template.jinja
│   ├── special_tokens_map.json
│   ├── adapter_config.json
│   └── modeling_nemotron_h.py
├── optim
│   ├── __0_0.distcp
├── rng
│   ├── rng_dp_rank_0.pt
```

In [ ]:
!python Automodel/examples/llm_finetune/finetune.py -c nemotron_nano_v3_squad_peft.yaml

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'froze

**Sample Output**

```2026-01-16 01:36:06 | INFO | root | Experiment_details:
2026-01-16 01:36:06 | INFO | root | Timestamp: '2026-01-16T01:36:06'
2026-01-16 01:36:06 | INFO | root | User: root
2026-01-16 01:36:06 | INFO | root | Host: 043890d1df57
2026-01-16 01:36:06 | INFO | root | World size: 1
2026-01-16 01:36:06 | INFO | root | Backend: nccl
2026-01-16 01:36:06 | INFO | root | Recipe: TrainFinetuneRecipeForNextTokenPrediction
2026-01-16 01:36:06 | INFO | root | Model name: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
2026-01-16 01:36:06 | INFO | root | Recipe config:
2026-01-16 01:36:06 | INFO | root |   step_scheduler:
2026-01-16 01:36:06 | INFO | root |     global_batch_size: 8
2026-01-16 01:36:06 | INFO | root |     local_batch_size: 1
2026-01-16 01:36:06 | INFO | root |     ckpt_every_steps: 1000
2026-01-16 01:36:06 | INFO | root |     val_every_steps: 1000
2026-01-16 01:36:06 | INFO | root |     max_steps: 100


Fetching 13 files:   0%|                                 | 0/13 [00:00<?, ?it/s]
model-00001-of-00013.safetensors:   0%|             | 0.00/4.99G [00:00<?, ?B/s]
model-00010-of-00013.safetensors: 100%|████| 4.99G/4.99G [03:22<00:00, 24.6MB/s]
Fetching 13 files: 100%|████████████████████████| 13/13 [08:17<00:00, 38.29s/it]
Loading checkpoint shards: 100%|████████████████| 13/13 [00:02<00:00,  6.07it/s]
generation_config.json: 100%|██████████████████| 197/197 [00:00<00:00, 2.57MB/s]


2026-01-16 01:44:42 | INFO | root | Model summary:
2026-01-16 01:44:42 | INFO | root | --------------------------------
2026-01-16 01:44:42 | INFO | root | Trainable parameters: 220,968,448
2026-01-16 01:44:42 | INFO | root | Total parameters: 31,798,905,792
2026-01-16 01:44:42 | INFO | root | Trainable parameters percentage: 0.69%
2026-01-16 01:44:42 | INFO | root | Param L2 norm: 0.0000
2026-01-16 01:44:42 | INFO | root | --------------------------------
2026-01-16 01:44:43 | INFO | nemo_automodel.components.distributed.fsdp2 | World size is 1, skipping parallelization.

Generating train split: 100%|██| 87599/87599 [00:00<00:00, 643214.24 examples/s]
Generating validation split: 100%|█| 10570/10570 [00:00<00:00, 579678.26 example

2026-01-16 01:58:16 | INFO | root | step 14 | epoch 0 | loss 0.2154 | grad_norm 210.0000 | lr 9.93e-06 | mem 65.72 GiB | tps 47.12(47.12/gpu) | num_label_tokens 105
2026-01-16 02:44:53 | INFO | root | step 99 | epoch 0 | loss 0.0533 | grad_norm 40.7500 | lr 1.00e-06 | mem 65.49 GiB | tps 44.09(44.09/gpu) | num_label_tokens 97
```

## Run SFT

```python 
# for single process run
!python ../Automodel/examples/llm_finetune/finetune.py -c nemotron_nano_v3_squad.yaml
```

```python 
# for distributed run
!torchrun --nproc-per-node=8 ../Automodel/examples/llm_finetune/finetune.py -c nemotron_nano_v3_squad.yaml
```

## Publish the SFT Checkpoint or PEFT Adapters to the Hugging Face Hub
After fine-tuning a Hugging Face model using NeMo AutoModel, the resulting checkpoints or PEFT adapters are stored in a Hugging Face-native format, making it easy to share and deploy. To make these checkpoints and adapters publicly accessible, we can upload them to the Hugging Face Model Hub, allowing seamless integration with the Hugging Face ecosystem.

Using the Hugging Face Hub API, we can push the fine-tuned checkpoint or PEFT adapter to a repository, ensuring that others can easily load and use it with transformer's AutoModelForCausalLM for fine-tuned checkpoint, and peft.AutoPeftModel for PEFT adapters. The following steps outline how to publish the fine-tuned checkpoint or PEFT adapter:

1. Install the Hugging Face Hub library (if not already installed):
```python
pip3 install huggingface_hub
```
   
2. Log in to Hugging Face using your authentication token:
```python
hf auth login
```

3. Upload the fine-tuned checkpoint using the huggingface_hub Python API:
```python
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="checkpoints/epoch_0_step_10/model/consolidated",
    repo_id="your-username/llama3.2_1b-finetuned-name" or "your-username/peft-adapter-name",
    repo_type="model"
)
```
   
Once uploaded, the fine-tuned checkpoint can be loaded directly using:
```python
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("your-username/llama3.2_1b-finetuned-name")
```

Similarly, the PEFT adapter can be loaded directly using:

```python
from peft import PeftModel, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("base-model")
peft_model = PeftModel.from_pretrained(model, "your-username/peft-adapter-name")
```


## Useful Resources

- https://github.com/NVIDIA-NeMo/Automodel/discussions/976
- https://github.com/NVIDIA-NeMo/Automodel/blob/main/docs/guides/llm/finetune.md (slightly outdated but useful)
- https://github.com/NVIDIA-NeMo/Automodel/blob/main/examples/llm_finetune/nemotron/nemotron_nano_v3_squad.yaml 
- https://github.com/NVIDIA-NeMo/Automodel/blob/main/examples/llm_finetune/nemotron/nemotron_nano_v3_squad_peft.yaml
